# 🧪 pixelNeRF : Synthèse 3D à partir d'Une Seule Image (Single-View NVS)
**Projet Master M2I — De la 2D à la 3D : NeRF, pixelNeRF & LRM**

Ce notebook implémente l'inférence de **pixelNeRF (Yu et al., CVPR 2021)** à partir des poids pré-entraînés officiels :
1. **Étape 1 :** Vérification de l'environnement GPU
2. **Étape 2 :** Montage Google Drive
3. **Étape 3 :** Clonage du dépôt officiel pixelNeRF et installation des dépendances
4. **Étape 4 :** Téléchargement des checkpoints officiels (pré-entraînés sur ShapeNet)
5. **Étape 5 :** Téléchargement du jeu de données de test (ShapeNet SRN Chairs / Cars)
6. **Étape 6 :** Inférence 3D monoculaire (Génération d'une vidéo 360° à partir d'1 seule vue 2D)
7. **Étape 7 :** Calcul des métriques de reconstruction (PSNR, SSIM)
8. **Étape 8 :** Visualisation du rendu 3D et sauvegarde sur Google Drive

### Étape 1 : Vérification de l'accélérateur GPU (CUDA)

In [1]:
import torch
print("=== VÉRIFICATION DU GPU ===")
print(f"Version PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU actif : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ VRAM totale : {vram:.2f} Go")
else:
    raise SystemError("❌ Aucun GPU détecté. Allez dans Exécution -> Modifier le type d'exécution -> GPU T4.")

=== VÉRIFICATION DU GPU ===
Version PyTorch : 2.14.1+cu130
✅ GPU actif : Tesla T4
✅ VRAM totale : 14.56 Go


### Étape 2 : Montage de Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pixelnerf_results

Mounted at /content/drive


### Étape 3 : Installation du code officiel pixelNeRF et dépendances

In [3]:
import os
%cd /content
if not os.path.exists('/content/pixel-nerf'):
    !git clone https://github.com/sxyu/pixel-nerf.git

%cd /content/pixel-nerf
!pip install -q gdown scikit-image imageio imageio-ffmpeg torchvision
print("✅ Environnement pixelNeRF prêt !")

/content
Cloning into 'pixel-nerf'...
remote: Enumerating objects: 457, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 457 (delta 137), reused 120 (delta 120), pack-reused 274 (from 1)
Receiving objects: 100% (457/457), 2.78 MiB | 23.76 MiB/s, done.
Resolving deltas: 100% (235/235), done.
/content/pixel-nerf
✅ Environnement pixelNeRF prêt !


### Étape 4 : Téléchargement des checkpoints pré-entraînés officiels
*(Poids officiels fournis par les auteurs de UC Berkeley sur Google Drive)*

In [4]:
import os
%cd /content/pixel-nerf
!mkdir -p checkpoints

if not os.path.exists('/content/pixel-nerf/checkpoints/srn_chair.zip'):
    print("Téléchargement du checkpoint officiel pixelNeRF (ShapeNet Chairs)...")
    # Checkpoint officiel SRN Chairs
    !gdown --id 1UO_rL201guN6euoWkCOn-XpqR2e8o6ju -O /content/checkpoints_all.zip
    !unzip -q /content/checkpoints_all.zip -d /content/pixel-nerf/checkpoints/

print("✅ Checkpoints disponibles dans /content/pixel-nerf/checkpoints/ :")
!ls -la /content/pixel-nerf/checkpoints/

/content/pixel-nerf
Téléchargement du checkpoint officiel pixelNeRF (ShapeNet Chairs)...
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1UO_rL201guN6euoWkCOn-XpqR2e8o6ju
From (redirected): https://drive.google.com/uc?id=1UO_rL201guN6euoWkCOn-XpqR2e8o6ju&confirm=t&uuid=b6250c4f-c8e3-45fe-a56d-757bc3ceb071
To: /content/checkpoints_all.zip
100% 1.07G/1.07G [00:14<00:00, 74.9MB/s]
✅ Checkpoints disponibles dans /content/pixel-nerf/checkpoints/ :
total 28
drwxr-xr-x  7 root root 4096 Nov 25 15:57 .
drwxr-xr-x 12 root root 4096 Nov 25 15:57 ..
drwxr-x---  2 root root 4096 Jan  4  2021 dtu
drwxr-x---  2 root root 4096 Jan  4  2021 sn64
drwxr-x---  2 root root 4096 Jan  4  2021 sn64_unseen
drwxr-x---  2 root root 4096 Jan  4  2021 srn_car
drwxr-x---  2 root roo

### Étape 5 : Téléchargement d'un échantillon du dataset de test (ShapeNet SRN Chairs)

In [5]:
%cd /content/pixel-nerf
!mkdir -p /content/data/srn_chairs

# Téléchargement d'un mini-split de test SRN Chairs
print("Téléchargement des données de test SRN Chairs...")
!gdown --folder --id 1PsT3uKwqHHD2bEEHkIXB99AlIjtmrEiR -O /content/data/srn_data_raw
print("✅ Données de test prêtes !")

/content/pixel-nerf
Téléchargement des données de test SRN Chairs...
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Retrieving folder contents
Processing file 1aTSmJa8Oo2qCc2Ce2kT90MHEA6UTSBKj dtu_dataset.zip
Processing file 1Uxf0GguAUTSFIDD_7zuPbxk1C9WgXjce genlist.py
Processing file 1PFFPxO_641K4py8qC21mR4YjToz77AJT multi_chair_1v_checkpoint.zip
Processing file 1VV1ex0Prcy2mSs_GpZ4XbhP1vYLn3Z9s multi_chair_2v_checkpoint.zip
Processing file 11P3rsRjyh1QRdTAqWKVa7QUFaKOM3T10 multi_chair_test.zip
Processing file 1rQZHxjWvvI6OrZcKinCgRQuWJIcoaWXk multi_chair_train.zip
Processing file 1eRtF39XlrRghECBI7AYSr9eJxrGyA2rA multi_chair_val.zip
Processing file 19yDsEJjx9zNpOKz9o6AaK-E8ED6taJWU srn_cars.zip
Processing file 1VWtRZkC4_ON6EBhNNBTag-BwsjPB2Yc8 srn_chairs.zip
Retrieving folder contents completed
Building directory

In [13]:
!pip install dotmap pyhocon scikit-image imageio imageio-ffmpeg

In [11]:
# Corrige l'import torchvision dans data_util.py
!sed -i 's/import torchvision.transforms.functional_tensor as F_t/import torchvision.transforms.functional as F_t/g' /content/pixel-nerf/src/data/data_util.py
print("✅ Import torchvision corrigé !")

✅ Import torchvision corrigé !


In [15]:
!ls -la /content/data/
!find /content/data/ -maxdepth 3 -type d

total 16
drwxr-xr-x 4 root root 4096 Nov 25 15:57 .
drwxr-xr-x 1 root root 4096 Nov 25 15:57 ..
drwxr-xr-x 2 root root 4096 Nov 25 15:57 srn_chairs
drwxr-xr-x 2 root root 4096 Nov 25 15:59 srn_data_raw
/content/data/
/content/data/srn_data_raw
/content/data/srn_chairs


In [16]:
!ls -la /content/data/srn_data_raw/

total 9003172
drwxr-xr-x 2 root root       4096 Nov 25 15:59 .
drwxr-xr-x 4 root root       4096 Nov 25 15:57 ..
-rw-r--r-- 1 root root  953042617 Jan  2  2021 dtu_dataset.zip
-rw-r--r-- 1 root root       1370 Jan  3  2021 genlist.py
-rw-r--r-- 1 root root  215276175 Jan  4  2021 multi_chair_1v_checkpoint.zip
-rw-r--r-- 1 root root  215077657 Jan  4  2021 multi_chair_2v_checkpoint.zip
-rw-r--r-- 1 root root  373148457 Dec  3  2020 multi_chair_test.zip
-rw-r--r-- 1 root root 1073932948 Dec  3  2020 multi_chair_train.zip
-rw-r--r-- 1 root root  158774543 Dec  3  2020 multi_chair_val.zip
-rw-r--r-- 1 root root 2361708143 Jan  3  2021 srn_cars.zip
-rw-r--r-- 1 root root 3868241585 Jan  3  2021 srn_chairs.zip


In [17]:
print("Extraction du dataset SRN Chairs (split test)...")
# On extrait les données directement dans /content/data/
!unzip -q /content/data/srn_data_raw/srn_chairs.zip -d /content/data/

print("Vérification des dossiers extraits :")
!ls -d /content/data/*chair*

Extraction du dataset SRN Chairs (split test)...
Vérification des dossiers extraits :
/content/data/chairs_test   /content/data/chairs_val
/content/data/chairs_train  /content/data/srn_chairs


### Étape 6 : Inférence 3D à partir d'Une Seule Image (1-View NVS)
- `-n srn_chair` : utilise le modèle entraîné sur les chaises
- `-P 64` : conditionne la reconstruction sur la vue source numéro 64 uniquement (**1 seule image**)
- `-S 0` : indice du premier objet de test

In [18]:
%cd /content/pixel-nerf

!python eval/gen_video.py \
    -n srn_chair \
    --gpu_id 0 \
    --split test \
    -P '64' \
    -D /content/data/chairs \
    -S 0

/content/pixel-nerf
EXPERIMENT NAME: srn_chair
* Config file: conf/exp/srn.conf
* Dataset format: srn
* Dataset location: /content/data/chairs
Loading SRN dataset /content/data/chairs_test name: chairs
Data instance loaded: /content/data/chairs_test/103b75dfd146976563ed57e35c972b4b
Using torchvision resnet34 encoder
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet34-

### Étape 7 : Visualisation de la vidéo générée à partir d'une seule image

In [19]:
import glob
from IPython.display import Video, display

videos = sorted(glob.glob('/content/pixel-nerf/visuals/**/*.mp4', recursive=True))
if videos:
    print("Vidéo pixelNeRF 360° trouvée :", videos[-1])
    display(Video(videos[-1], embed=True, width=512))
else:
    !find /content/pixel-nerf/ -name "*.mp4"

Vidéo pixelNeRF 360° trouvée : /content/pixel-nerf/visuals/srn_chair/videot0000_v064.mp4


### Étape 8 : Sauvegarde des résultats pixelNeRF sur Google Drive

In [20]:
!mkdir -p /content/drive/MyDrive/pixelnerf_results/
!cp -r /content/pixel-nerf/visuals /content/drive/MyDrive/pixelnerf_results/
print("🎉 Vidéo 3D pixelNeRF 1-vue sauvegardée sur Google Drive !")

🎉 Vidéo 3D pixelNeRF 1-vue sauvegardée sur Google Drive !
